# SNP data access — `Ag3`

This notebook demonstrates the methods on the `Ag3` class that give access to raw and biallelic SNP (single nucleotide polymorphism) data: site masks, SNP calls, allele counts, site annotations, genome accessibility, biallelic SNP subsets (including LD-pruned and PLINK-exported versions), and diplotype encodings. Each section shows the method's parameters and one worked, executed example.

In [1]:
import malariagen_data
ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../../gcs_cache"),
    results_cache="../../results_cache",
)
ag3

/opt/homebrew/Caskroom/miniconda/base/envs/malariagen2/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


<MalariaGEN Ag3 API client>
Storage URL                           : simplecache::gs://vo_agam_release_master_us_central1
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
Results cache                         : /Users/katie.barr/malariagen-data-python/results_cache
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.8.0.post13+b769b728
Client location                       : England, United Kingdom
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.8.0.post13+b769b728/Ag3.html

## `site_mask_ids`

A read-only property (no parameters, called without parentheses) that returns the identifiers of the site-filter masks available for this data resource. These identifiers are the valid values for the `site_mask` parameter accepted by most other SNP-data methods (`snp_calls`, `biallelic_snp_calls`, `is_accessible`, etc.). For *Ag3* there are three masks, each defined by a different combination of taxa used to build the site filter: `gamb_colu_arab` (pass in gambiae, coluzzii *and* arabiensis — the strictest, most widely-applicable mask), `gamb_colu` (pass in gambiae and coluzzii only), and `arab` (pass in arabiensis only). Choosing a narrower mask retains more sites that are reliable for that subset of taxa but may include sites that are unreliable in others.

In [2]:
ag3.site_mask_ids

('gamb_colu_arab', 'gamb_colu', 'arab')

## `snp_calls`

Access SNP sites, site filters and genotype calls for a genome region and set of samples, returned as a lazy `xarray.Dataset` (genotype data are backed by dask arrays, so nothing large is actually read from storage until you compute/subset it). Parameters:

- **region**: genome region(s) to query — a contig name, a region string `"{contig}:{start}-{end}"`, a gene/transcript ID, or a list of these. Narrowing the region reduces the number of variant sites returned.
- **sample_sets**: which sample set(s) and/or release(s) to include. Restricting to fewer sample sets reduces the `samples` dimension.
- **sample_query**: a pandas query string evaluated against the sample metadata to select which samples to include, e.g. `"taxon == 'coluzzii'"`. Mutually exclusive with `sample_indices`.
- **sample_query_options**: extra keyword arguments passed through to pandas `query()`/`eval()` (e.g. `engine`, `parser`) when evaluating `sample_query`.
- **sample_indices**: an alternative to `sample_query` — an explicit list of positional sample indices (as ordered in the sample metadata) to select. Cannot be combined with `sample_query`.
- **site_mask**: which site-filters mask (see `site_mask_ids`) to apply; `None` disables site filtering entirely, returning every site in the region including unreliable ones.
- **site_class**: restrict to sites of one functional class (e.g. `CDS_DEG_4` for 4-fold degenerate coding sites, `INTRON_SHORT`, `UTR_5PRIME`, `INTERGENIC`, …); `None` (default) returns all sites regardless of class.
- **inline_array**: passed through to dask's `from_array()`; a performance/graph-construction knob with no effect on the data returned.
- **chunks**: how the underlying zarr data are divided into dask chunks — `"native"` (default, use the on-disk zarr chunking), a target size string such as `"300 MiB"`, `"auto"`, or an explicit chunk tuple. Affects memory usage/parallelism, not the values returned.
- **cohort_size**: if set, randomly down-sample the selected samples to exactly this many (raises an error if there are fewer than this).
- **min_cohort_size**: raise an error if fewer than this many samples remain after selection — a safety check rather than a down-sample.
- **max_cohort_size**: randomly down-sample only if *more* than this many samples are selected (no error if fewer).
- **random_seed**: seed for the random down-sampling performed by `cohort_size`/`max_cohort_size`, so results are reproducible.

The example below restricts to *coluzzii* samples from one sample set in a 1 Mbp region of 3L, applies the strictest site mask, and caps the cohort at 20 samples for a smaller, reproducible example.

In [3]:
ds_snps = ag3.snp_calls(
    region="3L:15,000,000-16,000,000",
    sample_sets="AG1000G-BF-A",
    sample_query="taxon == 'coluzzii'",
    site_mask="gamb_colu_arab",
    max_cohort_size=20,
    random_seed=42,
)
ds_snps

Load sample metadata: ⠋ (0:00:00.00)

Access SNP calls: ⠋ (0:00:00.00)

Access SNP calls: ⠙ (0:00:00.09)

Access SNP calls: ⠹ (0:00:00.18)

Access SNP calls: ⠸ (0:00:00.27)

...

Access SNP calls: ⠙ (0:15:08.15)

Access SNP calls: ⠹ (0:15:08.24)

Apply site filters: ⠋ (0:00:00.00)

/Users/katie.barr/malariagen-data-python/malariagen_data/anoph/snp_data.py:1185: UserWarning: Cohort downsampled from 82 to 20 samples. Set max_cohort_size=None to disable downsampling.
  return self._snp_calls(


<xarray.Dataset> Size: 185MB
Dimensions:                             (variants: 558410, alleles: 4,
                                         samples: 20, ploidy: 2)
Coordinates:
    variant_position                    (variants) int32 2MB dask.array<chunksize=(340807,), meta=np.ndarray>
    variant_contig                      (variants) uint8 558kB dask.array<chunksize=(340807,), meta=np.ndarray>
    sample_id                           (samples) <U24 2kB dask.array<chunksize=(20,), meta=np.ndarray>
Dimensions without coordinates: variants, alleles, samples, ploidy
Data variables:
    variant_allele                      (variants, alleles) |S1 2MB dask.array<chunksize=(340807, 4), meta=np.ndarray>
    variant_filter_pass_gamb_colu_arab  (variants) bool 558kB dask.array<chunksize=(138463,), meta=np.ndarray>
    variant_filter_pass_gamb_colu       (variants) bool 558kB dask.array<chunksize=(138463,), meta=np.ndarray>
    variant_filter_pass_arab            (variants) bool 558kB dask.array<chunksize=(138463,), meta=np.ndarray>
    call_genotype                       (variants, samples, ploidy) int8 22MB dask.array<chunksize=(138463, 20, 2), meta=np.ndarray>
    call_GQ                             (variants, samples) int16 22MB dask.array<chunksize=(138463, 20), meta=np.ndarray>
    call_MQ                             (variants, samples) int16 22MB dask.array<chunksize=(138463, 20), meta=np.ndarray>
    call_AD                             (variants, samples, alleles) int16 89MB dask.array<chunksize=(138463, 20, 4), meta=np.ndarray>
    call_genotype_mask                  (variants, samples, ploidy) bool 22MB dask.array<chunksize=(138463, 20, 2), meta=np.ndarray>
Attributes:
    contigs:  ('2R', '2L', '3R', '3L', 'X')

## `snp_allele_counts`

Compute the number of times each of the 4 possible alleles (reference plus 3 alternates) is observed at each SNP site, across the selected samples. This triggers real computation over genotype data (results are cached in the `results_cache` directory for re-use). Parameters `region`, `sample_sets`, `sample_query`, `sample_query_options`, `sample_indices`, `site_mask`, `site_class`, `cohort_size`, `min_cohort_size`, `max_cohort_size`, `random_seed`, `inline_array` and `chunks` have the same meaning as for `snp_calls` above. One extra parameter:

- **return_dataset**: if `False` (default), return a plain numpy array of shape `(n_variants, 4)`. If `True`, return an `xarray.Dataset` containing the SNP calls with `variant_allele_count` added as an extra data variable — useful when you also want the variant coordinates attached to the counts.

Here we compute allele counts for the *coluzzii* cohort in the same region as above, and ask for the Dataset form so the result carries the variant positions alongside the counts.

In [4]:
ac_ds = ag3.snp_allele_counts(
    region="3L:15,000,000-16,000,000",
    sample_sets="AG1000G-BF-A",
    sample_query="taxon == 'coluzzii'",
    site_mask="gamb_colu_arab",
    return_dataset=True,
)
ac_ds

<xarray.Dataset> Size: 14MB
Dimensions:               (variants: 558410, alleles: 4)
Coordinates:
    variant_position      (variants) int32 2MB 15000000 15000001 ... 15999966
    variant_contig        (variants) uint8 558kB 3 3 3 3 3 3 3 ... 3 3 3 3 3 3 3
Dimensions without coordinates: variants, alleles
Data variables:
    variant_allele        (variants, alleles) |S1 2MB b'G' b'A' ... b'T' b'G'
    variant_allele_count  (variants, alleles) int32 9MB 21 143 0 0 ... 164 0 0 0

## `plot_snps`

Plot SNPs in a genome region as a two-track Bokeh figure: a SNPs track (segregating and non-segregating SNPs shown as rectangles on different levels, coloured by site filter) stacked above a genes track. Internally this composes `plot_snps_track` and `plot_genes`. Parameters:

- **region**, **sample_sets**, **sample_query**, **sample_query_options**, **site_mask**, **cohort_size**: as above — these control which SNPs are shown (`site_mask="default"` uses the resource's default mask).
- **sizing_mode**: Bokeh layout sizing mode (e.g. `"stretch_width"` default, `"fixed"`, `"scale_width"`) controlling how the plot resizes in the notebook/browser.
- **width**: plot width in pixels; `None` (default) lets `sizing_mode` control width.
- **track_height**: height in pixels of the SNPs track.
- **genes_height**: height in pixels of the genes track underneath.
- **max_snps**: maximum number of SNPs to render (default 200,000) — protects against trying to draw an unreasonable number of rectangles for very large regions.
- **show**: if `True` (default), display the figure immediately; if `False`, return the Bokeh figure object without displaying it.
- **gene_labels**: an optional mapping of gene ID to custom label text to show in the genes track.
- **gene_labelset**: an optional pre-built Bokeh `LabelSet` to use for gene labels instead of the default.

The example plots a small region with a down-sampled cohort of 10 samples, which keeps the number of visible SNPs manageable.

In [5]:
ag3.plot_snps(
    region="3R:28,550,000-28,650,000",
    sample_sets="AG1000G-BF-A",
    cohort_size=10,
)

Load genome features: ⠋ (0:00:00.00)

Load genome features: ⠙ (0:00:00.09)

Load genome features: ⠹ (0:00:00.19)

Load genome features: ⠸ (0:00:00.37)

GridPlot(id='p1110', ...)

## `site_annotations`

Access positional/functional annotations (codon position, codon degeneracy, sequence class such as CDS/intron/UTR, and relative position within the enclosing feature) for every SNP site in a region, as an `xarray.Dataset` aligned one-to-one with the SNP sites returned by `snp_calls` for the same region/mask. This is the raw annotation data that `site_class` filtering (in `snp_calls` etc.) is built on top of. Parameters:

- **region**: genome region to annotate (single region only, unlike the multi-region `regions` parameter used elsewhere).
- **site_mask**: optional site-filters mask to restrict to accessible sites before returning annotations; `None` (default) returns annotations for every SNP site regardless of filter status.
- **inline_array**, **chunks**: dask performance/chunking knobs, as above.

**Diagram opportunity:** a labelled diagram of a gene's structure (5' UTR, CDS with codon positions/degeneracy, introns with splice regions, 3' UTR, up/downstream/intergenic) mapped onto the `site_class` categories, so readers can see at a glance which `site_class` value corresponds to which part of a gene. Would fit well here or alongside `snp_calls`'s `site_class` parameter.

In [6]:
ds_ann = ag3.site_annotations(
    region="AGAP004707",
    site_mask="gamb_colu_arab",
)
ds_ann

Load genome features: ⠋ (0:00:00.00)

Load genome features: ⠙ (0:00:00.10)

Load genome features: ⠹ (0:00:00.19)

Load genome features: ⠸ (0:00:00.37)

<xarray.Dataset> Size: 580kB
Dimensions:           (variants: 36228)
Dimensions without coordinates: variants
Data variables:
    codon_degeneracy  (variants) int8 36kB dask.array<chunksize=(36228,), meta=np.ndarray>
    codon_nonsyn      (variants) uint8 36kB dask.array<chunksize=(36228,), meta=np.ndarray>
    codon_position    (variants) int8 36kB dask.array<chunksize=(36228,), meta=np.ndarray>
    seq_cls           (variants) uint8 36kB dask.array<chunksize=(36228,), meta=np.ndarray>
    seq_flen          (variants) uint32 145kB dask.array<chunksize=(36228,), meta=np.ndarray>
    seq_relpos_start  (variants) uint32 145kB dask.array<chunksize=(36228,), meta=np.ndarray>
    seq_relpos_stop   (variants) uint32 145kB dask.array<chunksize=(36228,), meta=np.ndarray>

## `is_accessible`

Compute a genome accessibility array for a region: a boolean numpy array with one value per *base pair* in the region (not just SNP sites), `True` where the site passes the chosen site-filters mask. Parameters:

- **region**: single genome region to compute accessibility for.
- **site_mask**: which site-filters mask defines "accessible" — `"default"` (the resource default) or any value from `site_mask_ids`. A stricter mask marks more positions as inaccessible.
- **inline_array**, **chunks**: dask performance knobs, as above.

The example computes accessibility across a small gene region using the `gamb_colu_arab` mask and reports the fraction of accessible bases.

In [7]:
is_acc = ag3.is_accessible(region="AGAP004707", site_mask="gamb_colu_arab")
is_acc, is_acc.mean()

Load genome features: ⠋ (0:00:00.00)

(array([ True,  True,  True, ...,  True,  True,  True]),
 np.float64(0.4931663490334876))

## `biallelic_snp_calls`

Like `snp_calls`, but restricted to sites that are biallelic *within the selected samples* (exactly 2 observed alleles), with unobserved alleles squeezed out so the `alleles` dimension is always 2. This is the entry point for most downstream analyses (PCA, LD pruning, PLINK export) that require a biallelic encoding. In addition to the `region`, `sample_sets`, `sample_query`, `sample_query_options`, `sample_indices`, `site_mask`, `site_class`, `inline_array`, `chunks`, `cohort_size`, `min_cohort_size`, `max_cohort_size` and `random_seed` parameters (same meaning as `snp_calls`), there are extra parameters for filtering and thinning the biallelic site set:

- **min_minor_ac**: minimum minor-allele count required to keep a site; sites with a rarer minor allele are dropped. Can be a float, interpreted as a minimum minor-allele *frequency* instead of a count.
- **max_missing_an**: maximum number of missing allele calls tolerated at a site (0 = require complete calls); can also be a float fraction. Sites with more missingness are dropped.
- **n_snps**: target number of SNPs to keep — if more sites remain after filtering, they are evenly thinned down to approximately this count; raises an error if fewer than `n_snps` sites remain.
- **thin_offset**: starting offset used by the thinning applied when `n_snps` is set, letting you pick a different (still evenly-spaced) subset of SNPs by changing the offset.

**Diagram opportunity:** a simple illustration of biallelic vs. multiallelic sites — e.g. one column of genotype calls showing a site with only reference/alt-1 observed (biallelic, kept) next to a site where alt-1 *and* alt-2 are both observed among samples (multiallelic, dropped by `biallelic_snp_calls`). Would fit right here, before the code example.

The example below applies both a minimum minor allele count and a missingness cap, illustrating non-default filtering values.

In [8]:
ds_bi = ag3.biallelic_snp_calls(
    region="3L:15,000,000-16,000,000",
    sample_sets="AG1000G-BF-A",
    site_mask="gamb_colu_arab",
    min_minor_ac=2,
    max_missing_an=20,
    chunks="300 MiB",
)
ds_bi

Access SNP calls: ⠋ (0:00:00.00)

Access SNP calls: ⠙ (0:00:00.09)

Apply site filters: ⠋ (0:00:00.00)

Prepare biallelic SNP calls: ⠋ (0:00:00.00)

Prepare biallelic SNP calls: ⠙ (0:00:00.14)

<xarray.Dataset> Size: 29MB
Dimensions:               (variants: 76735, alleles: 2, samples: 181, ploidy: 2)
Coordinates:
    variant_contig        (variants) uint8 77kB dask.array<chunksize=(46188,), meta=np.ndarray>
    variant_position      (variants) int32 307kB dask.array<chunksize=(46188,), meta=np.ndarray>
    sample_id             (samples) <U24 17kB dask.array<chunksize=(181,), meta=np.ndarray>
Dimensions without coordinates: variants, alleles, samples, ploidy
Data variables:
    variant_allele        (variants, alleles) |S1 153kB dask.array<chunksize=(46188, 2), meta=np.ndarray>
    variant_allele_count  (variants, alleles) int32 614kB 23 339 360 ... 3 360 2
    call_genotype         (variants, samples, ploidy) int8 28MB dask.array<chunksize=(66111, 50, 2), meta=np.ndarray>
Attributes:
    contigs:  ('2R', '2L', '3R', '3L', 'X')

## `biallelic_snp_calls_ld_pruned`

Obtains biallelic SNP calls (as `biallelic_snp_calls` above, after thinning to `n_snps`) and then prunes out SNPs that are in strong linkage disequilibrium with each other, using scikit-allel's `locate_unlinked`. The result has the same structure as `biallelic_snp_calls` output, and is the recommended input for ADMIXTURE-style analyses or PLINK export where independent markers are required. Shares `region`, `sample_sets`, `sample_query`, `sample_query_options`, `sample_indices`, `site_mask`, `min_minor_ac`, `max_missing_an`, `random_seed`, `inline_array` and `chunks` with `biallelic_snp_calls`, plus `n_snps` and `thin_offset` (both required/used the same way as above). LD-pruning-specific parameters:

- **ld_window_size**: number of SNPs in the sliding window used to compute pairwise r² (default 500). A larger window considers linkage over more SNPs at the cost of speed.
- **ld_window_step**: number of SNPs the window advances each iteration (default 200). Smaller steps give more thorough but slower pruning.
- **ld_threshold**: maximum r² allowed between a pair of SNPs before one is removed as linked (default 0.1). Lower values prune more aggressively, retaining fewer, more independent SNPs.

**Diagram opportunity:** a small flowchart showing the relationship between `biallelic_snp_calls` → `biallelic_snp_calls_ld_pruned` → `biallelic_snps_to_plink`/PCA, i.e. how thinning (`n_snps`) happens before LD pruning, and how the sliding window (`ld_window_size`, `ld_window_step`) moves across SNPs comparing pairs against `ld_threshold`. Would fit well here.

The example below compares a strict and a lenient `ld_threshold` on the same thinned SNP set, to show how many more SNPs survive pruning when linkage is tolerated more.

In [9]:
ds_pruned_strict = ag3.biallelic_snp_calls_ld_pruned(
    region="3L:15,000,000-16,000,000",
    n_snps=2_000,
    sample_sets="AG1000G-BF-A",
    ld_threshold=0.1,
)
ds_pruned_lenient = ag3.biallelic_snp_calls_ld_pruned(
    region="3L:15,000,000-16,000,000",
    n_snps=2_000,
    sample_sets="AG1000G-BF-A",
    ld_threshold=0.5,
)
print(f"strict  (threshold=0.1): {ds_pruned_strict.sizes['variants']} variants")
print(f"lenient (threshold=0.5): {ds_pruned_lenient.sizes['variants']} variants")
ds_pruned_strict

Prepare biallelic SNP calls: ⠋ (0:00:00.00)

Computing genotype ref counts:   0%|          | 0/81 [00:00<?, ?it/s]

LD pruning: ⠋ (0:00:00.00)

Prepare biallelic SNP calls: ⠋ (0:00:00.00)

Computing genotype ref counts:   0%|          | 0/81 [00:00<?, ?it/s]

LD pruning: ⠋ (0:00:00.00)

LD pruning: ⠙ (0:00:00.09)

strict  (threshold=0.1): 753 variants
lenient (threshold=0.5): 1856 variants


<xarray.Dataset> Size: 301kB
Dimensions:               (variants: 753, alleles: 2, samples: 181, ploidy: 2)
Coordinates:
    variant_contig        (variants) uint8 753B dask.array<chunksize=(452,), meta=np.ndarray>
    variant_position      (variants) int32 3kB dask.array<chunksize=(452,), meta=np.ndarray>
    sample_id             (samples) <U24 17kB dask.array<chunksize=(181,), meta=np.ndarray>
Dimensions without coordinates: variants, alleles, samples, ploidy
Data variables:
    variant_allele        (variants, alleles) |S1 2kB dask.array<chunksize=(452, 2), meta=np.ndarray>
    variant_allele_count  (variants, alleles) int32 6kB 23 339 360 ... 3 315 47
    call_genotype         (variants, samples, ploidy) int8 273kB dask.array<chunksize=(194, 50, 2), meta=np.ndarray>
Attributes:
    contigs:  ('2R', '2L', '3R', '3L', 'X')

## `biallelic_diplotypes`

Load biallelic SNP genotypes encoded as diplotypes: for each site and sample, the count of alternate alleles carried (0, 1 or 2), rather than the raw pair of allele calls. This is a compact numeric encoding often used as direct input to statistical/ML methods. It shares all the sample-selection, site-filtering, and thinning parameters of `biallelic_snp_calls` (`region`, `sample_sets`, `sample_query`, `sample_query_options`, `sample_indices`, `site_mask`, `site_class`, `cohort_size`, `min_cohort_size`, `max_cohort_size`, `random_seed`, `min_minor_ac`, `max_missing_an`, `n_snps`, `thin_offset`, `inline_array`, `chunks`), plus:

- **return_dataset**: if `False` (default), returns a tuple `(gn, samples)` where `gn` is a `(variants, samples)` array of alternate allele counts. If `True`, returns an `xarray.Dataset` with `call_diplotype` plus `sample_id`, `variant_position` and `variant_contig` coordinates attached.

The example below thins to 2,000 SNPs and requests the tuple form (the default, and the form most directly usable with numpy/scikit-learn style code).

In [10]:
gn, samples = ag3.biallelic_diplotypes(
    region="3L:15,000,000-16,000,000",
    sample_sets="AG1000G-BF-A",
    site_mask="gamb_colu_arab",
    min_minor_ac=2,
    n_snps=2_000,
)
gn.shape, samples[:5]

((2036, 181),
 array(['AB0085-Cx', 'AB0086-Cx', 'AB0087-C', 'AB0088-C', 'AB0089-Cx'],
       dtype='<U24'))

## `biallelic_snps_to_plink`

Write biallelic SNP data (as selected by `biallelic_snp_calls` internally) to the PLINK binary file format (`.bed`/`.bim`/`.fam`). Shares the region/sample-selection/thinning/filtering parameters with `biallelic_snp_calls` (`region`, `n_snps`, `thin_offset`, `sample_sets`, `sample_query`, `sample_query_options`, `sample_indices`, `site_mask`, `min_minor_ac`, `max_missing_an`, `random_seed`, `inline_array`, `chunks`), plus:

- **output_dir**: directory in which to write the PLINK files.
- **overwrite**: if `False` (default), an existing file with the same auto-generated name is left untouched and its path is returned without recomputation; set `True` to force regeneration.
- **out**: an explicit output file prefix. If not given, a prefix is auto-generated from the SNP selection parameters (region, n_snps, min_minor_ac, max_missing_an, thin_offset), so different parameter combinations don't collide.

The method returns the base path; append `.bed`, `.bim` or `.fam` to get the individual file paths. The example writes a small 2,000-SNP PLINK dataset to a `plink_output` subfolder next to this notebook.

In [11]:
import os

plink_prefix = ag3.biallelic_snps_to_plink(
    output_dir="plink_output",
    region="3L:15,000,000-16,000,000",
    n_snps=2_000,
    sample_sets="AG1000G-BF-A",
    overwrite=True,
)
plink_prefix, sorted(os.listdir("plink_output"))

Prepare biallelic SNP calls: ⠋ (0:00:00.00)

Computing genotype ref counts:   0%|          | 0/81 [00:00<?, ?it/s]

Prepare output data: ⠋ (0:00:00.00)

('plink_output/3L:15,000,000-16,000,000.2000.2.0.0',
 ['3L:15,000,000-16,000,000.2000.2.0.0.bed',
  '3L:15,000,000-16,000,000.2000.2.0.0.bim',
  '3L:15,000,000-16,000,000.2000.2.0.0.fam'])